In [1]:
# =====================================================================
# HINDI HANDWRITTEN WORD RECOGNITION – mBART PIPELINE
# TPS-STN + Swin + BiLSTM + mBART-50 Decoder + LLRD + XLM-R Verifier
# =====================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import cv2
import random
import unicodedata
import numpy as np
from collections import defaultdict

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

import timm

# mBART imports  (replaces GPT2LMHeadModel, GPT2Config)
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    XLMRobertaTokenizer,
    XLMRobertaForMaskedLM,
)

import Levenshtein
from jiwer import wer

# ============================================================
# 1. CONFIGURATION
# ============================================================
ROOT_DIR        = "/home/mca24-26/ocr/dataset/Word_Level_Hindi_Training_Set/Word_Level_Training_Set"
TRAIN_TXT       = os.path.join(ROOT_DIR, "train.txt")
CHECKPOINT_PATH = "./best_hindi_htr_mbart.pth"

IMG_HEIGHT  = 64
IMG_WIDTH   = 256
MAX_SEQ_LEN = 40
BEAM_SIZE   = 5

# mBART-large-50 uses d_model=1024
# BiLSTM hidden=512 → 512+512=1024 output  (matches mBART)
D_MODEL    = 1024
BATCH_SIZE = 8          # reduced: mBART decoder is large
LR         = 5e-5
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED        = 42
USE_AMP     = True

# ============================================================
# 2. SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============================================================
# 3. TOKENIZER  (replaces manual Hindi char vocab entirely)
#
# MBart50TokenizerFast supports Hindi natively.
# Every sequence must start with the language token: hi_IN
# The tokenizer handles Devanagari subwords — no char mapping needed.
# ============================================================
print("Loading mBART-50 tokenizer...")
tokenizer = MBart50TokenizerFast.from_pretrained(
    "facebook/mbart-large-50",
    src_lang="hi_IN",   # Hindi source language
    tgt_lang="hi_IN",   # Hindi target language
)
tokenizer.src_lang = "hi_IN"
tokenizer.tgt_lang = "hi_IN"

VOCAB_SIZE  = len(tokenizer)          # 250054
PAD_IDX     = tokenizer.pad_token_id  # 1
EOS_IDX     = tokenizer.eos_token_id  # 2
# mBART uses language token as BOS for the decoder
LANG_TOKEN_ID = tokenizer.convert_tokens_to_ids("hi_IN")  # e.g. 250010

print(f"VOCAB SIZE : {VOCAB_SIZE}")
print(f"PAD={PAD_IDX}  EOS={EOS_IDX}  LANG_TOKEN(BOS)={LANG_TOKEN_ID}")


def encode_text(text):
    """
    Tokenize Hindi text for mBART decoder.
    Returns label ids with PAD replaced by -100 for loss masking.
    mBART decoder target format: [hi_IN, tok1, tok2, ..., EOS, PAD, PAD...]
    """
    enc = tokenizer(
        text,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    ids = enc.input_ids.squeeze(0)          # (MAX_SEQ_LEN,)
    # Replace PAD with -100 so CrossEntropyLoss ignores padding
    labels = ids.masked_fill(ids == PAD_IDX, -100)
    return labels


def decode_tokens(token_ids):
    """Decode mBART token ids back to Hindi text."""
    ids = [t for t in token_ids.tolist()
           if t not in [PAD_IDX, EOS_IDX, LANG_TOKEN_ID, -100]]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()


# ============================================================
# 4. IMAGE PREPROCESSING  (unchanged from your original)
# ============================================================
def preprocess_image(img_path):
    img  = cv2.imread(img_path)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Deskew
    gray   = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]
    edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180,
        threshold=40, minLineLength=30, maxLineGap=5
    )
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 != x1:
                angles.append(
                    np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
                )
        if angles:
            angle = np.median(angles)
    if abs(angle) > 20:
        angle = 0.0
    (h, w) = img.shape[:2]
    center    = (w // 2, h // 2)
    M         = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed  = cv2.warpAffine(
        img, M, (w, h),
        flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )

    # Resize + pad
    h, w  = deskewed.shape[:2]
    scale = IMG_HEIGHT / h
    new_w = int(w * scale)
    resized = cv2.resize(deskewed, (new_w, IMG_HEIGHT))
    if new_w < IMG_WIDTH:
        pad     = np.ones(
            (IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8
        ) * 255
        resized = np.concatenate([resized, pad], axis=1)
    else:
        resized = cv2.resize(resized, (IMG_WIDTH, IMG_HEIGHT))

    img_tensor = resized.astype(np.float32) / 255.0
    img_tensor = (img_tensor - 0.5) / 0.5
    img_tensor = np.transpose(img_tensor, (2, 0, 1))
    return torch.tensor(img_tensor, dtype=torch.float32)


# ============================================================
# 5. DATASET
# ============================================================
class HindiWordDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image  = preprocess_image(img_path)
        labels = encode_text(text)          # now uses mBART tokenizer
        return image, labels, text


# ============================================================
# 6. LOAD & SPLIT DATA  (unchanged)
# ============================================================
samples = []
with open(TRAIN_TXT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) != 2:
            continue
        rel_path, text = parts
        text     = unicodedata.normalize("NFC", text.strip())
        img_path = os.path.join(ROOT_DIR, rel_path)
        if os.path.exists(img_path):
            samples.append((img_path, text))

print("TOTAL SAMPLES:", len(samples))

train_samples, temp_samples = train_test_split(
    samples, test_size=0.15, random_state=SEED
)
val_samples, test_samples = train_test_split(
    temp_samples, test_size=0.5, random_state=SEED
)
print(f"TRAIN:{len(train_samples)}  VAL:{len(val_samples)}  TEST:{len(test_samples)}")

train_loader = DataLoader(
    HindiWordDataset(train_samples), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    HindiWordDataset(val_samples), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    HindiWordDataset(test_samples), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)


# ============================================================
# 7. ARCHITECTURE
# ============================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B   = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        cp             = self.localization(x)
        cx             = cp[:, :, 0].mean(dim=1)
        cy             = cp[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(
            x, grid, align_corners=False, padding_mode='border'
        )


class SwinEncoder(nn.Module):
    """
    Swin-B stage3 output: 512 channels
    Projected to D_MODEL=1024 to match mBART decoder d_model
    """
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        # 512 → 1024  (mBART needs 1024)
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]           # stage 3: (B, H, W, 512)
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))   # (B, H*W, 1024)


# ============================================================
# mBART DECODER
#
# What we do:
#   1. Load full facebook/mbart-large-50
#   2. Extract only model.decoder + lm_head  (discard encoder)
#   3. Feed our Swin+BiLSTM memory as encoder_hidden_states
#      into the mBART decoder's cross-attention layers
#   4. Prepend language token (hi_IN) as BOS for every sequence
#
# Why it works:
#   mBART decoder cross-attention expects (B, seq, 1024)
#   Our BiLSTM outputs (B, T, 1024)  → dimensions match directly
# ============================================================
class MBartDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        print("Loading facebook/mbart-large-50 ...")
        mbart = MBartForConditionalGeneration.from_pretrained(
            "facebook/mbart-large-50"
        )
        # Keep only the decoder stack and language model head
        self.decoder = mbart.model.decoder   # MBartDecoder (transformer stack)
        self.lm_head = mbart.lm_head         # Linear(1024 → 250054)
        # Final layer norm (part of mBART model, needed before lm_head)
        self.final_layer_norm = mbart.model.shared  # embedding shared weights

        # We directly use mbart's decoder — no need to rebuild config
        # mBART decoder d_model is already 1024
        del mbart
        print("mBART decoder ready. d_model=1024, vocab=250054")

    def forward(self, input_ids, encoder_hidden_states):
        """
        input_ids            : (B, tgt_seq)   decoder input token ids
        encoder_hidden_states: (B, src_seq, 1024)  visual memory from BiLSTM
        returns logits       : (B, tgt_seq, vocab_size)
        """
        out    = self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
        )
        # out.last_hidden_state: (B, tgt_seq, 1024)
        logits = self.lm_head(out.last_hidden_state)
        return logits   # (B, tgt_seq, 250054)


class CompleteHTRPipeline(nn.Module):
    def __init__(self, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)

        # BiLSTM: hidden=512, bidirectional → output=1024 = mBART d_model
        self.bilstm = nn.LSTM(
            input_size    = d_model,      # 1024
            hidden_size   = d_model // 2, # 512
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.mbart_decoder = MBartDecoder()

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)   # (B, T, 1024)
        memory, _ = self.bilstm(swin_feat)         # (B, T, 1024)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        """
        target_ids: (B, MAX_SEQ_LEN) with -100 at PAD positions
        Teacher forcing: decoder sees [LANG_TOK, tok1, ..., tokN-1]
                         and predicts  [tok1,     tok2, ..., tokN  ]
        """
        memory = self._extract_visual_memory(images)  # (B, T, 1024)

        # Build decoder input: replace -100 with PAD, then shift right
        dec_input = target_ids.clone()
        dec_input[dec_input == -100] = PAD_IDX

        # Shift right: prepend language token as BOS, drop last token
        lang_tok  = torch.full(
            (dec_input.size(0), 1), LANG_TOKEN_ID,
            dtype=torch.long, device=dec_input.device
        )
        dec_input = torch.cat([lang_tok, dec_input[:, :-1]], dim=1)
        # dec_input shape: (B, MAX_SEQ_LEN)

        logits = self.mbart_decoder(
            input_ids=dec_input,
            encoder_hidden_states=memory
        )   # (B, MAX_SEQ_LEN, vocab_size)

        labels = target_ids.clone()   # -100 already in place

        loss_fn = criterion if criterion is not None else \
                  nn.CrossEntropyLoss(ignore_index=-100)
        return loss_fn(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN, beam_size=1):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)

        # Greedy decoding (beam_size=1)
        # Start every sequence with the language token (acts as BOS)
        generated = torch.full(
            (B, 1), LANG_TOKEN_ID,
            dtype=torch.long, device=device
        )
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_length - 1):
            logits      = self.mbart_decoder(
                input_ids=generated,
                encoder_hidden_states=memory
            )
            next_tokens = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            next_tokens = next_tokens.masked_fill(
                finished.unsqueeze(1), EOS_IDX
            )
            generated = torch.cat([generated, next_tokens], dim=1)
            finished |= (next_tokens.squeeze(1) == EOS_IDX)
            if finished.all():
                break

        # Strip leading language token
        return generated[:, 1:]

    @torch.no_grad()
    def generate_beam(self, images, max_length=MAX_SEQ_LEN,
                      beam_size=BEAM_SIZE):
        """Beam search — used only at test time."""
        device      = images.device
        B           = images.size(0)
        memory      = self._extract_visual_memory(images)
        all_results = []

        for b in range(B):
            mem       = memory[b:b+1]
            beams     = [(0.0, [LANG_TOKEN_ID])]
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                        continue
                    inp     = torch.tensor(
                        [tokens], dtype=torch.long, device=device
                    )
                    logits  = self.mbart_decoder(
                        input_ids=inp,
                        encoder_hidden_states=mem
                    )
                    log_prob = F.log_softmax(
                        logits[0, -1], dim=-1
                    )
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(
                        top_lp.tolist(), top_id.tolist()
                    ):
                        candidates.append(
                            (score + lp, tokens + [tid])
                        )

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break
                if not beams:
                    break

            pool             = completed if completed else beams
            _, best_tokens   = max(pool, key=lambda x: x[0])
            result           = torch.tensor(
                best_tokens[1:],   # strip language token
                dtype=torch.long, device=device
            )
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full(
            (B, max_len), EOS_IDX,
            dtype=torch.long, device=device
        )
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded


# ============================================================
# 8. LLRD OPTIMIZER
# mBART decoder groups replace gpt2 groups
# Swin + BiLSTM + TPS groups are unchanged
# ============================================================
def build_llrd_optimizer(model, base_lr=LR,
                         decay_factor=0.75, weight_decay=0.05):
    assigned = set()

    def collect(named_params, filter_fn):
        params = []
        for name, param in named_params:
            if id(param) not in assigned and filter_fn(name):
                params.append(param)
                assigned.add(id(param))
        return params

    def collect_all(named_params):
        params = []
        for name, param in named_params:
            if id(param) not in assigned:
                params.append(param)
                assigned.add(id(param))
        return params

    # mBART cross-attention layers (pretrained, fine-tune faster)
    mbart_crossattn = collect(
        model.mbart_decoder.named_parameters(),
        lambda n: "encoder_attn" in n   # mBART names cross-attn "encoder_attn"
    )
    # mBART lm_head
    mbart_lmhead = collect(
        model.mbart_decoder.named_parameters(),
        lambda n: "lm_head" in n
    )
    # Rest of mBART decoder (self-attn + FFN layers)
    mbart_rest    = collect_all(model.mbart_decoder.named_parameters())
    bilstm_params = collect_all(model.bilstm.named_parameters())
    swin_proj = collect(
        model.swin_encoder.named_parameters(),
        lambda n: n.startswith("proj.") or n.startswith("norm.")
    )
    swin_s4 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_3" in n
    )
    swin_s3 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_2" in n
    )
    swin_s2 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_1" in n
    )
    swin_s1   = collect_all(model.swin_encoder.swin.named_parameters())
    tps_params = collect_all(model.tps_stn.named_parameters())

    lr = [
        base_lr,
        base_lr * decay_factor,
        base_lr * decay_factor**2,
        base_lr * decay_factor**3,
        base_lr * decay_factor**4,
        base_lr * decay_factor**5,
        base_lr * decay_factor**6,
        base_lr * decay_factor**7,
        base_lr * decay_factor**3,
        base_lr * decay_factor**4,
    ]

    groups = [
        (mbart_crossattn, lr[0], "mbart_crossattn"),
        (mbart_lmhead,    lr[1], "mbart_lmhead"),
        (mbart_rest,      lr[2], "mbart_rest"),
        (bilstm_params,   lr[3], "bilstm"),
        (swin_proj,       lr[4], "swin_proj"),
        (swin_s4,         lr[5], "swin_stage4"),
        (swin_s3,         lr[6], "swin_stage3"),
        (swin_s2,         lr[7], "swin_stage2"),
        (swin_s1,         lr[8], "swin_stage1"),
        (tps_params,      lr[9], "tps_stn"),
    ]

    param_groups = [
        {"params": p, "lr": l, "name": n}
        for p, l, n in groups if len(p) > 0
    ]

    print("\nLLRD Groups:")
    print(f"{'Name':<20} {'LR':>10} {'Params':>10}")
    print("-" * 44)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<20} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    return AdamW(param_groups, weight_decay=weight_decay)


# ============================================================
# 9. AGENTIC VERIFICATION — XLM-RoBERTa MLM scorer
#    Two-stage: fast Levenshtein filter → XLM-R re-ranking
# ============================================================
class AgenticVerificationModule:
    def __init__(self, train_samples, device=DEVICE):
        # Stage 1: frequency lexicon (fast filter)
        self.lexicon  = defaultdict(int)
        for _, text in train_samples:
            for word in text.split():
                clean = word.strip(".,!?()[]{}:;\"'")
                if clean:
                    self.lexicon[clean] += 1
        self.freq_max = max(self.lexicon.values()) if self.lexicon else 1
        self.device   = device
        print(f"Lexicon built: {len(self.lexicon)} words")

        # Stage 2: XLM-RoBERTa MLM scorer
        print("Loading XLM-RoBERTa verifier...")
        self.xlm_tok   = XLMRobertaTokenizer.from_pretrained(
            "xlm-roberta-base"
        )
        self.xlm_model = XLMRobertaForMaskedLM.from_pretrained(
            "xlm-roberta-base"
        ).to(device).eval()
        print("XLM-RoBERTa ready.")

    @torch.no_grad()
    def _xlm_score(self, word):
        """Score word by MLM log-probability in a Hindi context."""
        context = f"यह {word} है"
        inputs  = self.xlm_tok(
            context, return_tensors="pt"
        ).to(self.device)
        out      = self.xlm_model(**inputs)
        logprobs = torch.log_softmax(out.logits[0], dim=-1)
        word_toks = self.xlm_tok.tokenize(word)
        word_ids  = self.xlm_tok.convert_tokens_to_ids(word_toks)
        scores = [
            logprobs[i + 1, wid].item()
            for i, wid in enumerate(word_ids)
            if i + 1 < logprobs.size(0)
        ]
        return sum(scores) / max(len(scores), 1)

    def verify_and_correct(self, text_output,
                           confidence=None,
                           confidence_threshold=0.85):
        cleaned = text_output.strip()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if confidence is not None and confidence >= confidence_threshold:
            return text_output

        # Stage 1: Levenshtein candidates
        candidates = []
        target_len = len(cleaned)
        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 2:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2:
                continue
            candidates.append((word, freq, dist))

        if not candidates:
            return text_output

        # Stage 2: XLM-R re-ranking
        best, best_score = text_output, -float('inf')
        for word, freq, dist in candidates:
            xlm_s  = self._xlm_score(word)
            freq_s = freq / self.freq_max
            score  = xlm_s + freq_s - (dist * 0.5)
            if score > best_score:
                best_score = score
                best       = word

        # Preserve casing
        if text_output.isupper():
            return best.upper()
        elif text_output and text_output[0].isupper():
            return best.capitalize()
        return best


# ============================================================
# 10. METRICS & EARLY STOPPING  (unchanged)
# ============================================================
def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)


def calculate_metrics(preds, targets):
    cer     = char_accuracy(preds, targets)
    wer_val = wer(targets, preds) * 100
    return {"CER": cer, "WER": wer_val}


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# ============================================================
# 11. TRAINING LOOP
# ============================================================
def train():
    model = CompleteHTRPipeline().to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    print(f"Total parameters : {total/1e6:.2f}M")
    print(f"  Swin+BiLSTM    : encoder stack")
    print(f"  mBART decoder  : ~610M (decoder only)")

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False

    optimizer = build_llrd_optimizer(
        model, base_lr=LR, decay_factor=0.75, weight_decay=0.05
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    criterion      = nn.CrossEntropyLoss(
        ignore_index=-100, label_smoothing=0.1
    )
    scaler         = GradScaler("cuda", enabled=USE_AMP)
    agent_verifier = AgenticVerificationModule(train_samples)
    early_stopper  = EarlyStopping(
        patience=EARLY_STOPPING_PATIENCE, min_delta=0.1
    )
    best_val_wer   = float('inf')
    swin_unfrozen  = False

    for epoch in range(1, MAX_EPOCHS + 1):

        # Unfreeze Swin at epoch 4
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = build_llrd_optimizer(
                model, base_lr=LR,
                decay_factor=0.75, weight_decay=0.05
            )
            scheduler = torch.optim.lr_scheduler\
                .CosineAnnealingWarmRestarts(
                    optimizer, T_0=10, T_mult=2, eta_min=1e-7
                )
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        # ---- Train ----
        model.train()
        train_loss = 0.0
        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch}/{MAX_EPOCHS} [Train]"
        )
        for images, targets, _ in pbar:
            images  = images.to(DEVICE)
            targets = targets.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda", enabled=USE_AMP):
                loss = model(
                    images, target_ids=targets,
                    criterion=criterion
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)

        # ---- Validation (greedy) ----
        model.eval()
        all_preds, all_labels = [], []
        first_batch_done      = False

        with torch.no_grad():
            for images, _, texts in tqdm(val_loader, desc="Val"):
                images = images.to(DEVICE)
                tokens = model.generate(images, beam_size=1)
                preds  = [decode_tokens(x) for x in tokens]
                verified = [
                    agent_verifier.verify_and_correct(p)
                    for p in preds
                ]
                all_preds.extend(verified)
                all_labels.extend(texts)

                if not first_batch_done:
                    print("\n--- Sample predictions ---")
                    for i in range(min(3, len(preds))):
                        print(
                            f"Target: '{texts[i]}' | "
                            f"Pred: '{preds[i]}' | "
                            f"Verified: '{verified[i]}'"
                        )
                    first_batch_done = True

        metrics  = calculate_metrics(all_preds, all_labels)
        val_cer  = metrics["CER"]
        val_wer  = metrics["WER"]
        print(
            f"\nEpoch {epoch} | Loss: {avg_train_loss:.4f} "
            f"| CER: {val_cer:.2f}% | WER: {val_wer:.2f}%"
        )

        if val_wer < best_val_wer:
            best_val_wer = val_wer
            torch.save({
                'epoch':            epoch,
                'model_state_dict': model.state_dict(),
                'best_wer':         val_wer,
                'cer':              val_cer,
                'decoder':          'mbart-large-50',
            }, CHECKPOINT_PATH)
            print(f"Best model saved (WER: {val_wer:.2f}%)")

        if early_stopper(val_wer):
            print("Early stopping triggered.")
            break

    print("Training finished.")


# ============================================================
# 12. TEST EVALUATION
# ============================================================
def test(use_beam_search=True):
    model = CompleteHTRPipeline().to(DEVICE)
    if os.path.exists(CHECKPOINT_PATH):
        ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(
            f"Loaded epoch {ckpt.get('epoch','?')} "
            f"WER={ckpt.get('best_wer','?'):.2f}%"
        )
    else:
        print("No checkpoint found.")
        return

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, _, texts in tqdm(test_loader, desc="Test"):
            images = images.to(DEVICE)
            if use_beam_search:
                tokens = model.generate_beam(
                    images, beam_size=BEAM_SIZE
                )
            else:
                tokens = model.generate(images, beam_size=1)
            preds = [decode_tokens(x) for x in tokens]
            test_preds.extend(preds)
            test_labels.extend(texts)

    test_cer = char_accuracy(test_preds, test_labels)
    test_wer = wer(test_labels, test_preds) * 100
    print(f"\nTest CER: {test_cer:.2f}% | Test WER: {test_wer:.2f}%")
    for i in range(min(10, len(test_preds))):
        print(f"GT : {test_labels[i]}")
        print(f"PR : {test_preds[i]}")
        print("-" * 40)


# ============================================================
# 13. MAIN
# ============================================================
if __name__ == "__main__":
    train()
    test(use_beam_search=True)

Loading mBART-50 tokenizer...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


VOCAB SIZE : 250054
PAD=1  EOS=2  LANG_TOKEN(BOS)=250010
TOTAL SAMPLES: 85585
TRAIN:72747  VAL:6419  TEST:6419


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading facebook/mbart-large-50 ...
mBART decoder ready. d_model=1024, vocab=250054
Total parameters : 559.13M
  Swin+BiLSTM    : encoder stack
  mBART decoder  : ~610M (decoder only)

LLRD Groups:
Name                         LR     Params
--------------------------------------------
mbart_crossattn        5.00e-05     50.41M
mbart_rest             2.81e-05    408.26M
bilstm                 2.11e-05     12.60M
swin_proj              1.58e-05      0.53M
swin_stage4            1.19e-05     27.30M
swin_stage3            8.90e-06     57.30M
swin_stage2            6.67e-06      1.71M
swin_stage1            2.11e-05      0.40M
tps_stn                1.58e-05      0.63M
Lexicon built: 6943 words
Loading XLM-RoBERTa verifier...


...


Epoch 19 | Loss: 1.6093 | CER: 91.39% | WER: 8.46%
EarlyStopping: 8/8
Early stopping triggered.
Training finished.
Loading facebook/mbart-large-50 ...
mBART decoder ready. d_model=1024, vocab=250054


/tmp/ipykernel_1137383/775999078.py:887: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Loaded epoch 12 WER=7.73%


Test: 100%|███████████████████████████████████████████| 803/803 [3:56:05<00:00, 17.64s/it]


Test CER: 91.15% | Test WER: 8.74%
GT : प्रवाह
PR : प्रवाह
----------------------------------------
GT : अधिक
PR : अधिक
----------------------------------------
GT : प्रदान
PR : प्रदान
----------------------------------------
GT : तक
PR : तक
----------------------------------------
GT : स्थापित
PR : स्थापित
----------------------------------------
GT : है
PR : है
----------------------------------------
GT : पहले
PR : पहले
----------------------------------------
GT : ।
PR : ।
----------------------------------------
GT : इसे
PR : इसे
----------------------------------------
GT : का
PR : का
----------------------------------------
